# 10.1.1. 희소 표현 기반 임베딩

In [2]:
import pandas as pd
class2 = pd.read_csv('C:\_SY\pytorch_prac\DLPytorchTextbook\chap10\data\class2.csv')

In [7]:
class2.head()

,Unnamed: 0,id,tissue,class,class2,x,y,r
0,0,CID000,C,CIRC,N,535.0,475.0,192.0
1,1,CID001,A,CIRA,N,433.0,268.0,58.0
2,2,CID002,A,CIRA,I,NaN,NaN,NaN
3,3,CID003,C,CIRC,B,NaN,NaN,NaN
4,4,CID004,F,CIRF,I,488.0,145.0,29.0


In [5]:
from sklearn import preprocessing
label_encoder = preprocessing.LabelEncoder()
onehot_encoder = preprocessing.OneHotEncoder()

In [8]:
train_x = label_encoder.fit_transform(class2['class2'])
train_x

array([2, 2, 1, 0, 1, 0])

# 10.1.2. 횟수 기반 임베딩

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    'This is last chance.',
    'and if you do not have this chance',
    'you will never get any chance',
    'will you do get this one?',
    'please, get this chance',
]

vect = CountVectorizer()
vect.fit(corpus)
vect.vocabulary_

{'this': 13,
 'is': 7,
 'last': 8,
 'chance': 2,
 'and': 0,
 'if': 6,
 'you': 15,
 'do': 3,
 'not': 10,
 'have': 5,
 'will': 14,
 'never': 9,
 'get': 4,
 'any': 1,
 'one': 11,
 'please': 12}

In [11]:
vect.transform(['you will never get any chance.']).toarray()

array([[0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1]], dtype=int64)

In [12]:
vect = CountVectorizer(stop_words=["and", "is", "please", "this"]).fit(corpus)
vect.vocabulary_

{'last': 6,
 'chance': 1,
 'if': 5,
 'you': 11,
 'do': 2,
 'not': 8,
 'have': 4,
 'will': 10,
 'never': 7,
 'get': 3,
 'any': 0,
 'one': 9}

In [18]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

doc = ['I like machine learning', 'I love deep learning', 'I run everyday']

tfidf_vectorizer = TfidfVectorizer(min_df=1)
tfidf_matrix = tfidf_vectorizer.fit_transform(doc)
doc_distance = (tfidf_matrix * tfidf_matrix.T)

print('유사도를 위한', str(doc_distance.get_shape()[0]), 'x', str(doc_distance.get_shape()[1]), '행렬을 만들었습니다.')
print(doc_distance.toarray())

유사도를 위한 3 x 3 행렬을 만들었습니다.
[[1.       0.224325 0.      ]
 [0.224325 1.       0.      ]
 [0.       0.       1.      ]]


# 10.1.3. 예측 기반 임베딩

In [22]:
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jenny\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [23]:
from nltk.tokenize import sent_tokenize, word_tokenize
import warnings
warnings.filterwarnings(action='ignore')
import gensim
from gensim.models import Word2Vec

sample = open("C:\_SY\pytorch_prac\DLPytorchTextbook\chap10\data\peter.txt", "r", encoding='UTF-8')
s = sample.read()

f = s.replace("\n", " ")
data = []

for i in sent_tokenize(f): # 문장 먼저 자르고
    temp = []
    for j in word_tokenize(i): # 단어로 다른다음
        temp.append(j.lower()) # 소문자로 바꿔서
    data.append(temp) # 넣기

data

[['once',
  'upon',
  'a',
  'time',
  'in',
  'london',
  ',',
  'the',
  'darlings',
  'went',
  'out',
  'to',
  'a',
  'dinner',
  'party',
  'leaving',
  'their',
  'three',
  'children',
  'wendy',
  ',',
  'jhon',
  ',',
  'and',
  'michael',
  'at',
  'home',
  '.'],
 ['after',
  'wendy',
  'had',
  'tucked',
  'her',
  'younger',
  'brothers',
  'jhon',
  'and',
  'michael',
  'to',
  'bed',
  ',',
  'she',
  'went',
  'to',
  'read',
  'a',
  'book',
  '.'],
 ['she', 'heard', 'a', 'boy', 'sobbing', 'outside', 'her', 'window', '.'],
 ['he', 'was', 'flying', '.'],
 ['there', 'was', 'little', 'fairy', 'fluttering', 'around', 'him', '.'],
 ['wendy', 'opened', 'the', 'window', 'to', 'talk', 'to', 'him', '.'],
 ['“', 'hello', '!'],
 ['who', 'are', 'you', '?'],
 ['why', 'are', 'you', 'crying', '”', ',', 'wendy', 'asked', 'him', '.'],
 ['“', 'my', 'name', 'is', 'peter', 'pan', '.'],
 ['my',
  'shadow',
  'wouldn',
  '’',
  't',
  'stock',
  'to',
  'me.',
  '”',
  ',',
  'he',
  'rep

## CBOW
- Continuous Bag of Words
- 문장에서 등장하는 n개 단어 열에서 다음에 등장할 단어 예측

In [30]:
# 데이터셋에 CBOW 적용 후 peter와 wendy의 유사성 확인
model1 = gensim.models.Word2Vec(data, # CBOW 적용할 데이터셋
                                min_count=1, # 단어의 최소 빈도수 제한
                                vector_size=300, # 임베딩된 벡터의 차원
                                window=5, # 컨텍스트 윈도우 크기
                                sg=0) # sg=0 일떈 CBOW, 1일떈 skip-gram, default는 CBOW
print("Cosine similarity between peter " + "wendy - CBOW : ",
      model1.wv.similarity('peter', 'wendy'))

# Cosine similarity between peter wendy - CBOW :  0.074393824
# word2vec이 무작위로 초기화되고, 훈련 과정에서도 무작위 처리되므로 달라질 수 있음.
# 근데 차이가 너무 많이 나는데...? 이럴수가 있나?

print("Cosine similarity between peter " + "hook - CBOW : ",
      model1.wv.similarity('peter', 'hook'))
# Cosine similarity between peter hook - CBOW :  0.02770986

Cosine similarity between peter wendy - CBOW :  0.005605979
Cosine similarity between peter hook - CBOW :  0.12987992


## Skip-gram
- CBOW와 반대로 특정 단어에서 문맥이 될 수 있는 단어 예측
- 가령 slept라는 단어를 이용하여 앞 뒤 단어 유추

In [32]:
model2 = gensim.models.Word2Vec(data, min_count=1, vector_size=100, window=5, sg =1)

print("Cosine similarity between peter & wendy - Skim gram", 
      model2.wv.similarity('peter', 'wendy'))

print("Cosine similarity between peter & hood - Skim gram", 
      model2.wv.similarity('peter', 'hook'))

Cosine similarity between peter & wendy - Skim gram 0.4008868
Cosine similarity between peter & hood - Skim gram 0.52016735


## 패스트텍스트
- Word2Vec의 한계 : 사전에 없는 단어에 대해서는 학습이 불안정함
    - 왜냐하면 워드임베딩 방식이 Distribited Representation(분산 표현)이라서,      
      단어의 분산 분포가 유사한 단어들에 비슷한 벡터값을 할당하여 표현하기 때문
- 그래서 패스트 텍스트는 Word Representation 방식을 사용함.     
  (사전에 없는 단어에 벡터 값을 부여)

In [33]:
from gensim.test.utils import common_texts
from gensim.models import FastText

model = FastText("C:\_SY\pytorch_prac\DLPytorchTextbook\chap10\data\peter.txt", vector_size=4, window=3, min_count=1, epochs=10)

sim_score1 = model.wv.similarity('peter', 'wendy')
print(sim_score1)

sim_score2 = model.wv.similarity('peter', 'hook')
print(sim_score2)

0.4592452
0.043825716


In [36]:
from __future__ import print_function
from gensim.models import KeyedVectors

model_kr = KeyedVectors.load_word2vec_format('C:\_SY\pytorch_prac\DLPytorchTextbook\chap10\data\wiki.ko.vec')

In [37]:
find_similar_to = '노력'

for similar_word in model_kr.similar_by_word(find_similar_to):
    print("word: {0}, simil: {1:.2f}".format(
        similar_word[0], similar_word[1]
    ))

word: 노력함, simil: 0.80
word: 노력중, simil: 0.75
word: 노력만, simil: 0.72
word: 노력과, simil: 0.71
word: 노력의, simil: 0.69
word: 노력가, simil: 0.69
word: 노력이나, simil: 0.69
word: 노력없이, simil: 0.68
word: 노력맨, simil: 0.68
word: 노력보다는, simil: 0.68


In [39]:
similarities = model_kr.most_similar(positive = ['동물', '육식동물'], negative=['사람'])
print(similarities)

[('초식동물', 0.7804121971130371), ('거대동물', 0.7547270059585571), ('육식동물의', 0.7547166347503662), ('유두동물', 0.753511369228363), ('반추동물', 0.7470757961273193), ('독동물', 0.7466291785240173), ('육상동물', 0.7460315823554993), ('유즐동물', 0.7450904250144958), ('극피동물', 0.7449344396591187), ('복모동물', 0.742434561252594)]


# 10.1.4. 횟수/예측 기반 임베딩
## Glove
- Global Vectors for Word Represenation
- 횟수 기반 LSA(Latent Semantic Analysis)와 예측 기반의 Word2Vec 단점을 보완하기 위한 모델
- 단어의 글로벌 동시 발생 확률 정보를 포함하는 임베딩 방법
- 단어에 대한 통계 정보 + Skip-gram

In [43]:
import numpy as np
%matplotlib notebook
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from sklearn.decomposition import PCA
from gensim.test.utils import datapath, get_tmpfile
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec

glove_file = datapath('C:\_SY\pytorch_prac\DLPytorchTextbook\chap10\data\glove.6B.100d.txt')
Word2Vec_glove_file = get_tmpfile('glove.6B.100d.word2vec.txt') # Glove 데이터를 word2vec 형태로 변환
glove2word2vec(glove_file, Word2Vec_glove_file)

(400000, 100)

In [44]:
model = KeyedVectors.load_word2vec_format(Word2Vec_glove_file)
model.most_similar('bill')

[('legislation', 0.8072139620780945),
 ('proposal', 0.7306863069534302),
 ('senate', 0.7142541408538818),
 ('bills', 0.704440176486969),
 ('measure', 0.6958035230636597),
 ('passed', 0.6906244158744812),
 ('amendment', 0.6846879720687866),
 ('provision', 0.6845567226409912),
 ('plan', 0.6816462874412537),
 ('clinton', 0.6663140654563904)]